In [2]:
import io
import zipfile
import requests
import frontmatter
from ingest import read_repo_data

In [3]:
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)

In [4]:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not filename.endswith('.md'):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [5]:
print(repository_data[1])

{'name': 'slack-faq-fetch', 'description': "Pull recent Slack channel discussions for a course (e.g. LLM Zoomcamp, Data Engineering Zoomcamp) into a review export to find missing FAQ entries. Determines the lookback window from the run log / git history, runs the fetcher, logs the run, and points to the export for review. Use when the user wants to fetch Slack questions, refresh FAQ content from Slack, or check what's missing for a course.", 'content': '# Slack FAQ Fetch\n\nFetch recent Slack discussions for a course so missing FAQ entries can be curated. The fetcher is an existing Python CLI (`faq_automation/slack_fetch.py`); this skill drives it end to end.\n\n## How it works\n\n`faq_automation.slack_fetch` calls the Slack Web API using the `SLACK_BOT_TOKEN` from `.env` (loaded automatically — **never print the token**), reads the course\'s `slack_channel` from `_questions/<course>/_metadata.yaml`, pulls the last `N` days of messages **plus thread replies**, and writes paired JSON + 

In [6]:
for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not (filename.endswith('.md') or filename.endswith('.mdx')):
        continue

    # rest remains the same...

In [7]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1374
Evidently documents: 95


In [8]:
from ingest import chunk_documents

In [9]:
chunked_dtc_faq = chunk_documents(dtc_faq)

In [30]:
from minsearch import Index
de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]


faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

In [31]:
query = 'What should be in a test dataset for AI evaluation?'
results = faq_index.search(query)

In [32]:
results

[{'id': '9f9a1b9e4f',
  'question': 'Terraform: Teardown of BigQuery Dataset',
  'sort_order': 13,
  'content': "When running `terraform destroy`, the following error can occur:\n\n```\nDo you really want to destroy all resources?\n\nTerraform will destroy all your managed infrastructure, as shown above.\n\nThere is no undo. Only 'yes' will be accepted to confirm.\n\nEnter a value: yes\n\ngoogle_bigquery_dataset.homework_dataset: Destroying... [id=projects/terraform-demo-449214/datasets/homework_dataset]\n\n╷\n\n│ Error: Error when reading or editing Dataset: googleapi: Error 400: Dataset terraform-demo-449214:homework_dataset is still in use, resourceInUse\n```\n\nThis is because the dataset is still in use by a table. To delete the dataset, set the `delete_contents_on_destroy` property to `true` in the `main.tf` file.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/module-1-terraform/013_9f9a1b9e4f_terraform-teardown-of-bigquery-dataset.md'},
 {'id': '8bfd724e4f',
  'q

In [16]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [33]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

In [34]:
query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

In [35]:
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)

  0%|          | 0/404 [00:00<?, ?it/s]

In [37]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [38]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)

In [39]:
query = 'Can I join the course now?'

text_results = faq_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = faq_vindex.search(q, num_results=5)

final_results = text_results + vector_results

In [40]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)

    return combined_results